### Imports


In [1]:
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage
from langchain.chains.llm import LLMChain 
from langchain.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory, FileChatMessageHistory
from dotenv import load_dotenv,  find_dotenv
load_dotenv('/home/lien/NLP/dj-fullstack-galore/app_rag/.env', 
            override=True)


LLM =  'gpt-4o-2024-08-06'  # "gpt-3.5-turbo" #
EMBEDDING_MODEL = 'text-embedding-3-large'
CHUNK_SIZE = 1000
CHUNK_OVERLAP=500
CHROMA_PATH = './chroma_db'

### Functions declaration

In [2]:
def load_document(file):
    '''
    Load file(s)
    '''
    import os
    _, extension = os.path.splitext(file)
    
    if extension == '.pdf':
        from langchain.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        from langchain.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    else:
        raise ValueError("Document format is not supported")
        
    data_from_file = loader.load()
    
    return data_from_file


def chunk_data(data, embedding_model, chunk_size=1000, chunk_overlap=0):
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                   chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(data) 
    
    # TODO:
    # from langchain_experimental.text_splitter import SemanticChunker
    # from langchain_openai.embeddings import OpenAIEmbeddings
    # text_splitter = SemanticChunker(OpenAIEmbeddings(model=embedding_model),
    #                                breakpoint_threshold_type="gradient")
    # chunks = text_splitter.create_documents([data])
        
    return chunks


def print_embedding_cost(texts):
    '''
    Calculate the OpenAI embedding costs
    '''
    import tiktoken
    enc = tiktoken.encoding_for_model(EMBEDDING_MODEL)
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost $: {0.0004 * total_tokens / 1000:.4f}')
    
    
def create_embeddings_chroma(chunks,
                             embedding_model=EMBEDDING_MODEL,
                             persist_directory='./chroma_db'):
    '''
    Use chroma db as vector store
    '''
    # from langchain_chroma import Chroma
    from langchain_community.vectorstores import Chroma
    from langchain_openai import ChatOpenAI
    from langchain_openai import OpenAIEmbeddings
    
    embedding_function = OpenAIEmbeddings(model=embedding_model)
    # chroma vector store object
    vector_store = Chroma.from_documents(chunks,
                                         embedding_function,
                                         persist_directory=persist_directory)
    return vector_store

def load_embeddings_chroma(persist_directory='./chroma_db'):
    '''
    Load the existing embeddings to a vector store object
    '''
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings
    
    embedding_function = OpenAIEmbeddings(model=model_name)
    
    vector_store = Chroma(persist_directory=persist_directory,
                          embedding_function=embedding_function)
    
    return vector_store

   

### LLM and ChromaDB setup

In [3]:
llm = ChatOpenAI(model=LLM, temperature=1)

# Transform loaders to Langchain data model
docs = load_document('/home/lien/NLP/dj-fullstack-galore/docs/PerDoc.pdf')

# Construct retriever ### 
 # https://chunkviz.up.railway.app/
 
splits = chunk_data(docs,
                    embedding_model=EMBEDDING_MODEL,
                    chunk_size=CHUNK_SIZE,
                    chunk_overlap=CHUNK_OVERLAP)

vectorstore = create_embeddings_chroma(splits,
                                     embedding_model=EMBEDDING_MODEL,
                                     persist_directory=CHROMA_PATH)
retriever = vectorstore.as_retriever()

print_embedding_cost(splits)

Loading /home/lien/NLP/dj-fullstack-galore/docs/PerDoc.pdf
Total Tokens: 197383
Embedding Cost $: 0.0790


### Contextualizing the questions

In [4]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# Chain with chat history
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

qa_system_prompt = """You are an assistant for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
If you don't know the answer, just say that you don't know. \
Use five sentences maximum and keep the answer concise.\ 
Answer all the questions in Swedish.\
{context}"""
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

### QA chaining interactive demo

In [5]:
import time
from langchain_core.messages import HumanMessage

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
chat_history = []

i = 1
print('Enter Quit or Exit to quit.')
while True:
    q = input(f'Question #{i}: ')
    i += 1
    if q.lower() in ['quit', 'exit']:
        print('Quitting...')
        time.sleep(2)
        print('User left the chat ... see you later, aligator!')
        break

    answer = rag_chain.invoke({"input": q, "chat_history": chat_history})
    chat_history.extend([HumanMessage(content=q), answer["answer"]])
    print(f'\nQuestion #{i-1}: {q}')
    print(f'\nAnswer: {answer["answer"]}\n')
    print("-" * 79)
    time.sleep(2)

Enter Quit or Exit to quit.

Question #1: what is the document about?

Answer: Dokumentet handlar om Naturvårdsverkets strategi och riktlinjer för bevarande av naturvärden i skogar och andra trädklädda marker, särskilt i skyddade områden. Det understryker vikten av att upprätthålla och återintroducera processer som är grundläggande för naturtypernas ekologiska funktionalitet. Strategin betonar långsiktigt positiva effekter på naturtypernas och de knutna arternas bevarandestatus, samt kortsiktiga positiva effekter. Det finns också information om relevanta regelverk, såsom områdesskyddslagstiftningen i miljöbalken. Dokumentet inkluderar även information om bevarandemål, målindikatorer och uppföljning av skyddade områden.

-------------------------------------------------------------------------------

Question #2: what are the natural types described in the document? 

Answer: Dokumentet beskriver främst naturtypen "landhöjningsskog" och "nordlig ädellövskog".

--------------------------